In [10]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()


google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

### The Illusion of "memory"

In [17]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Amit!"}
    ]
response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)
response.choices[0].message.content

"Hi Amit! It's great to meet you. How are you doing today? Is there anything I can help you with?"

### OK let's now ask a follow-up question

In [19]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)
response.choices[0].message.content
display(Markdown(response.choices[0].message.content))

I don't know your name! As an AI, I don't have access to your personal information or identity unless you choose to share it with me during our conversation. 

If you'd like me to know it, feel free to tell me!

## Maintaining history

In [38]:
from copy import deepcopy

messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

# User asks first question
messages.append({"role": "user", "content": "What is 10 + 20?"})

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)

answer = response.choices[0].message
response.choices[0].message

ChatCompletionMessage(content='10 + 20 = 30', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, extra_content={'google': {'thought_signature': 'EjQKMgERTTIPTo6QJMnZXwt/lZVKp/9u8/VjcoFm/oBHmt/jivQArv8pt2JjXRs+rQ6h5/8+'}})

In [39]:
display(Markdown(response.choices[0].message.content))

10 + 20 = 30

### Save a checkpoint

In [40]:
messages.append({
    "role": "assistant",
    "content": answer.content
})
checkpoint = deepcopy(messages)
checkpoint

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is 10 + 20?'},
 {'role': 'assistant', 'content': '10 + 20 = 30'}]

### Continue Branch A

In [41]:
messages.append({
    "role": "user",
    "content": "Multiply that by 5."
})

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)

messages.append(response.choices[0].message)
display(Markdown(response.choices[0].message.content))

30 * 5 = 150

In [42]:
messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is 10 + 20?'},
 {'role': 'assistant', 'content': '10 + 20 = 30'},
 {'role': 'user', 'content': 'Multiply that by 5.'},
 ChatCompletionMessage(content='30 * 5 = 150', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, extra_content={'google': {'thought_signature': 'EjQKMgERTTIPTn+ApdAtXQjjuhlCmxWup6Dc4qdbgNZj/l9+XyKjl55FRZeYukehYcCe7ah/'}})]

### Continue further

In [43]:
messages.append({
    "role": "user",
    "content": "Subtract 20."
})

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)

display(Markdown(response.choices[0].message.content))
messages.append(response.choices[0].message)
messages

150 - 20 = 130

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is 10 + 20?'},
 {'role': 'assistant', 'content': '10 + 20 = 30'},
 {'role': 'user', 'content': 'Multiply that by 5.'},
 ChatCompletionMessage(content='30 * 5 = 150', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, extra_content={'google': {'thought_signature': 'EjQKMgERTTIPTn+ApdAtXQjjuhlCmxWup6Dc4qdbgNZj/l9+XyKjl55FRZeYukehYcCe7ah/'}}),
 {'role': 'user', 'content': 'Subtract 20.'},
 ChatCompletionMessage(content='150 - 20 = 130', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, extra_content={'google': {'thought_signature': 'EjQKMgERTTIP3RWZATwcgTKSFIVleQtZgaEUALksX3LSWPrDrloxnxYQnGDZevnm71w85JDu'}})]

### Go back to the old conversation

In [44]:
messages = deepcopy(checkpoint)
messages

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user', 'content': 'What is 10 + 20?'},
 {'role': 'assistant', 'content': '10 + 20 = 30'}]

In [46]:
## From gemini we switched to gpt-4.1-mini for the next question, we still have the context of the previous conversation in the messages list.
messages.append({
    "role": "user",
    "content": "Multiply that by 8."
})

response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)

messages.append(response.choices[0].message)
response.choices[0].message
display(Markdown(response.choices[0].message.content))

240 multiplied by 8 equals 1,920.